# ARCADE Dataset Distribution Analysis

Analyses ground-truth masks in `arcade/masks/train` (or test/val) to answer:
- How many images contain each vessel class?
- Which class combinations co-occur in the same image?
- What is the per-class pixel count distribution?

In [ ]:
# ── Imports & configuration ───────────────────────────────────────────────────
import sys
from pathlib import Path
from itertools import combinations

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from PIL import Image
from tqdm.notebook import tqdm

REPO_ROOT = Path(".")
sys.path.insert(0, str(REPO_ROOT))

# ── Choose which split to analyse ─────────────────────────────────────────────
SPLIT       = "train"    # "train" | "val" | "test"
DIR_MASKS   = REPO_ROOT / "arcade" / "analysis" / SPLIT

N_CLASSES    = 4
CLASS_LABELS = ["Background", "LAD", "RCA", "LCX"]
FG_LABELS    = CLASS_LABELS[1:]          # foreground only
FG_INDICES   = list(range(1, N_CLASSES)) # [1, 2, 3]
FG_COLORS    = ["#e6194b", "#3cb44b", "#4363d8"]

IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
mask_files = sorted(p for p in DIR_MASKS.iterdir() if p.suffix.lower() in IMG_EXTS)

print(f"Split     : {SPLIT}")
print(f"Mask dir  : {DIR_MASKS}")
print(f"Images    : {len(mask_files)}")

## Scan Masks

In [ ]:
# ── Scan all masks once — collect per-image stats ────────────────────────────
# For each image:
#   pixel_counts[c]  — total pixels of class c
#   class_present[c] — True if class c has >= 1 pixel

records = []

for mp in tqdm(mask_files, desc="Scanning masks"):
    mask   = np.asarray(Image.open(mp).convert("L"), dtype=np.uint8)
    counts = np.bincount(mask.ravel(), minlength=N_CLASSES)   # shape (N_CLASSES,)
    present = counts > 0

    records.append({
        "name":    mp.stem,
        "counts":  counts,          # per-class pixel counts (includes background)
        "present": present,         # boolean array, length N_CLASSES
    })

print(f"Scanned {len(records)} images.")

# Aggregate
all_counts  = np.stack([r["counts"]  for r in records])   # (N, N_CLASSES)
all_present = np.stack([r["present"] for r in records])   # (N, N_CLASSES) bool

## 1 — Class Presence: How Many Images Contain Each Class?

A bar chart showing for each foreground class how many images contain at least one pixel of it.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

presence_counts = all_present[:, 1:].sum(axis=0)   # (3,) — n images with ≥1 pixel of each fg class
N = len(records)

bars = ax.bar(FG_LABELS, presence_counts, color=FG_COLORS, edgecolor="black", linewidth=0.8)
ax.set_xlabel("Vessel class")
ax.set_ylabel("Number of images")
ax.set_title(f"Class presence per image  (n={N}, split='{SPLIT}')")
ax.set_ylim(0, N * 1.1)
ax.axhline(N, color="grey", linewidth=0.8, linestyle="--", label=f"Total images ({N})")

for bar, cnt in zip(bars, presence_counts):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + N * 0.01,
        f"{cnt}\n({100*cnt/N:.1f}%)",
        ha="center", va="bottom", fontsize=10,
    )

ax.legend()
plt.tight_layout()
plt.show()

## 2 — Class Co-occurrence Across Images

A symmetric heatmap: cell (i, j) is the number of images in which class i **and** class j both appear.
The diagonal is therefore the same as the presence bar chart above.

In [ ]:
fg_present = all_present[:, 1:].astype(int)   # (N, 3) — LAD, RCA, LCX

# Build symmetric co-occurrence matrix
n_fg = len(FG_LABELS)
cooc = np.zeros((n_fg, n_fg), dtype=int)
for i in range(n_fg):
    for j in range(n_fg):
        cooc[i, j] = int((fg_present[:, i] & fg_present[:, j]).sum())

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cooc, cmap="YlOrRd", aspect="auto")

ax.set_xticks(range(n_fg)); ax.set_xticklabels(FG_LABELS)
ax.set_yticks(range(n_fg)); ax.set_yticklabels(FG_LABELS)
ax.set_title(f"Class co-occurrence (# images)  split='{SPLIT}'")

for i in range(n_fg):
    for j in range(n_fg):
        ax.text(j, i, str(cooc[i, j]), ha="center", va="center",
                fontsize=12, color="black" if cooc[i, j] < cooc.max() * 0.6 else "white")

plt.colorbar(im, ax=ax, label="# images")
plt.tight_layout()
plt.show()

## 3 — Per-Class Total Pixel Counts

**Left**: total pixel count per class summed across all images — shows class imbalance.  
**Right**: per-image pixel count distribution per class (box plot) — shows intra-class variability.

In [ ]:
ALL_LABELS = ["Background"] + FG_LABELS
ALL_COLORS = ["#cccccc"] + FG_COLORS

totals = all_counts.sum(axis=0)                # (4,) total pixels per class
grand_total = totals.sum()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Left: total pixel bar chart ───────────────────────────────────────────────
ax = axes[0]
bars = ax.bar(ALL_LABELS, totals, color=ALL_COLORS, edgecolor="black", linewidth=0.8, log=True)
ax.set_ylabel("Total pixel count (log scale)")
ax.set_title(f"Total pixels per class  split='{SPLIT}'")
for bar, tot in zip(bars, totals):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() * 1.3,
        f"{100*tot/grand_total:.2f}%",
        ha="center", va="bottom", fontsize=9,
    )

# ── Right: per-image box plot for fg classes only ─────────────────────────────
ax = axes[1]
fg_counts = [all_counts[:, c] for c in range(1, 4)]   # list of (N,) arrays

bp = ax.boxplot(
    fg_counts,
    patch_artist=True,
    medianprops=dict(color="black", linewidth=2),
    labels=FG_LABELS,
)
for patch, color in zip(bp["boxes"], FG_COLORS):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_yscale("log")
ax.set_ylabel("Pixel count per image (log scale)")
ax.set_title(f"Per-image fg pixel distribution  split='{SPLIT}'")

plt.tight_layout()
plt.show()

## 4 — Per-Image Pixel Breakdown (Stacked Bar)

Each vertical bar is one image. Segments show how many pixels belong to each class.
Images are sorted by total foreground pixel count (ascending) so outliers are visible at the right.

In [ ]:
fg_total_per_img = all_counts[:, 1:].sum(axis=1)   # (N,)
order = np.argsort(fg_total_per_img)               # ascending fg total
sorted_counts = all_counts[order]                  # (N, 4) reordered
x = np.arange(len(order))

fig, ax = plt.subplots(figsize=(max(10, len(order) // 5), 5))

bottom = np.zeros(len(order))
for c_idx, (label, color) in enumerate(zip(ALL_LABELS, ALL_COLORS)):
    ax.bar(x, sorted_counts[:, c_idx], bottom=bottom, label=label, color=color, width=1.0)
    bottom += sorted_counts[:, c_idx]

ax.set_xlabel("Image (sorted by total fg pixels)")
ax.set_ylabel("Pixel count")
ax.set_title(f"Per-image pixel breakdown  split='{SPLIT}'")
ax.set_xticks([])   # too many images to label
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()